# 04 — Exploration: stronger validation-set baselines

This notebook uses the same validation-pair construction and output contract as `02_inference.ipynb`: every model writes JPEGs plus a `results.csv` into its own `outputs/<model_name>/` folder.

**Models:**
- **E — Oracle target copy**: copies the validation target image. This is an upper bound / sanity check, not a fair deployable model.
- **F — Target color match**: keeps source/anchor structure but matches target-image color statistics. Also uses validation target information, so treat it as diagnostic.
- **G — SDXL img2img**: stronger image-to-image diffusion baseline.
- **H — SDXL ControlNet + Canny**: stronger structural diffusion baseline.

For `03_evaluate.ipynb`, add the model folders and suffixes from this notebook to its `MODELS` dictionary.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system diffusers==0.27.2 'transformers>=4.38.0,<5' 'huggingface-hub<0.26' accelerate controlnet-aux peft pillow-heif pandas opencv-python-headless
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# ── Config — edit these ──────────────────────────────────────────────────────
import os

if IN_COLAB:
    BASE = '/content/drive/My Drive/CIS_5190_group_project'
else:
    BASE = '..'

MANIFEST_CSV = f'{BASE}/manifest.csv'
ALIGNED_DIR = f'{BASE}/hf_dataset'

# Fast diagnostic baselines. These use target validation images, so they are upper-bound checks.
RUN_ORACLE_TARGET_COPY = True
RUN_TARGET_COLOR_MATCH = True

# Generative exploration models. These are heavier; enable on a GPU runtime.
RUN_SDXL_IMG2IMG = False
RUN_SDXL_CONTROLNET = False

# Diffusion hyperparams
SDXL_IMG2IMG_MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'
SDXL_CONTROLNET_MODEL = 'diffusers/controlnet-canny-sdxl-1.0'
SDXL_STRENGTH = 0.32
SDXL_GUIDANCE = 7.0
SDXL_STEPS = 30
SDXL_COND_SCALE = 0.65
CANNY_LOW = 80
CANNY_HIGH = 180

NEGATIVE = 'blurry, distorted, cartoon, painting, unrealistic, low quality, text, watermark'

print('Manifest:', MANIFEST_CSV)
print('Aligned dataset:', ALIGNED_DIR)

In [ ]:
# Construct the evaluation dataframe exactly like 02_inference.ipynb / 03_evaluate.ipynb.
import pandas as pd
from itertools import combinations
from pathlib import Path

mani_df = pd.read_csv(MANIFEST_CSV)
mani_df = mani_df[mani_df['status'] == 'kept']

meta_df = pd.read_csv(os.path.join(ALIGNED_DIR, 'metadata.csv'))
mani_df = mani_df.drop(columns=['caption'])

meta_df['fname'] = meta_df['file_name'].apply(lambda x: Path(x).name)
mani_df['fname'] = mani_df['file_name'].apply(lambda x: Path(x).stem) + '_aligned.jpg'
mani_df = mani_df.drop(columns=['file_name'])

mani_df = mani_df.merge(meta_df, on=['fname', 'location', 'time_of_day', 'weather', 'is_synthetic', 'split'], how='inner')
mani_df['anchor_file'] = 'images/' + mani_df['anchor_file'].apply(lambda x: Path(x).stem) + '_aligned_aligned.jpg'
mani_df = mani_df[mani_df['anchor_file'].isin(mani_df['file_name'])]
mani_df = mani_df[mani_df['split'] == 'val']

eval_df = {
    'src_name': [], 'location': [], 'src_tod': [], 'src_weather': [],
    'src_anchor': [],
    'tgt_name': [], 'tgt_tod': [], 'tgt_weather': [],
    'split': [],
}

for location in mani_df['location'].unique():
    loc_df = mani_df[mani_df['location'] == location].reset_index(drop=True)
    num_photos = len(loc_df)

    if num_photos == 1:
        eval_df['split'].append(loc_df['split'].iloc[0])
        eval_df['src_name'].append(loc_df['file_name'].iloc[0])
        eval_df['tgt_name'].append(loc_df['file_name'].iloc[0])
        eval_df['src_weather'].append(loc_df['weather'].iloc[0])
        eval_df['tgt_weather'].append(loc_df['weather'].iloc[0])
        eval_df['src_tod'].append(loc_df['time_of_day'].iloc[0])
        eval_df['tgt_tod'].append(loc_df['time_of_day'].iloc[0])
        eval_df['src_anchor'].append(loc_df['anchor_file'].iloc[0])
        eval_df['location'].append(location)
    else:
        for idx1, idx2 in combinations(list(range(num_photos)), 2):
            eval_df['split'].append(loc_df['split'].iloc[idx1])
            eval_df['src_name'].append(loc_df['file_name'].iloc[idx1])
            eval_df['tgt_name'].append(loc_df['file_name'].iloc[idx2])
            eval_df['src_weather'].append(loc_df['weather'].iloc[idx1])
            eval_df['tgt_weather'].append(loc_df['weather'].iloc[idx2])
            eval_df['src_tod'].append(loc_df['time_of_day'].iloc[idx1])
            eval_df['tgt_tod'].append(loc_df['time_of_day'].iloc[idx2])
            eval_df['location'].append(location)
            eval_df['src_anchor'].append(loc_df['anchor_file'].iloc[idx1])

eval_df = pd.DataFrame(eval_df)
print(f'Eval set: {len(eval_df)} pairs')
eval_df.head()

In [ ]:
import gc
import shutil

import cv2
import numpy as np
import torch
from PIL import Image

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def image_path(rel_path: str) -> str:
    return os.path.join(ALIGNED_DIR, rel_path)

def load_anchor(row) -> Image.Image:
    return Image.open(image_path(row['src_anchor'])).convert('RGB').resize((512, 512))

def load_target(row) -> Image.Image:
    return Image.open(image_path(row['tgt_name'])).convert('RGB').resize((512, 512))

def prompt_for(row) -> str:
    loc = row['location'].replace('_', ' ').title()
    tod = row['tgt_tod'].lower()
    weather = row['tgt_weather'].lower()
    return (
        f'A realistic architectural photo of {loc} on the University of Pennsylvania campus, '
        f'{tod}, {weather} weather, same viewpoint, same building geometry, natural lighting, high detail'
    )

def get_canny(img: Image.Image) -> Image.Image:
    arr = np.array(img.convert('L'))
    edges = cv2.Canny(arr, CANNY_LOW, CANNY_HIGH)
    return Image.fromarray(np.stack([edges] * 3, axis=-1))

def generated_name(row, suffix: str) -> str:
    stem = os.path.splitext(os.path.basename(row['tgt_name']))[0]
    return f'{stem}{suffix}'

def run_inference(pipe_fn, out_dir: str, suffix: str):
    os.makedirs(out_dir, exist_ok=True)
    results = []
    for _, row in eval_df.iterrows():
        anchor = load_anchor(row)
        out_img = pipe_fn(row, anchor).convert('RGB')
        fname = generated_name(row, suffix)
        out_img.save(os.path.join(out_dir, fname), quality=95)
        results.append({**row.to_dict(), 'generated_file': fname})
    pd.DataFrame(results).to_csv(os.path.join(out_dir, 'results.csv'), index=False)
    print(f'Saved {len(results)} images -> {out_dir}')

## E — Oracle target copy

This should score near-perfectly in `03_evaluate.ipynb`. Use it to confirm the evaluator and validation-pair plumbing are working.

In [ ]:
if RUN_ORACLE_TARGET_COPY:
    out_dir = f'{BASE}/outputs/oracle_target_copy'
    suffix = '_oracle.jpg'
    os.makedirs(out_dir, exist_ok=True)
    results = []

    for _, row in eval_df.iterrows():
        fname = generated_name(row, suffix)
        shutil.copy2(image_path(row['tgt_name']), os.path.join(out_dir, fname))
        results.append({**row.to_dict(), 'generated_file': fname})

    pd.DataFrame(results).to_csv(os.path.join(out_dir, 'results.csv'), index=False)
    print(f'Saved {len(results)} images -> {out_dir}')

## F — Target color match

A fast diagnostic baseline: preserve anchor structure while matching the target image's RGB mean and standard deviation.

In [ ]:
def match_color_stats(source: Image.Image, reference: Image.Image) -> Image.Image:
    src = np.asarray(source).astype(np.float32)
    ref = np.asarray(reference.resize(source.size)).astype(np.float32)

    src_mean = src.reshape(-1, 3).mean(axis=0)
    src_std = src.reshape(-1, 3).std(axis=0) + 1e-6
    ref_mean = ref.reshape(-1, 3).mean(axis=0)
    ref_std = ref.reshape(-1, 3).std(axis=0) + 1e-6

    matched = (src - src_mean) / src_std * ref_std + ref_mean
    matched = np.clip(matched, 0, 255).astype(np.uint8)
    return Image.fromarray(matched)

if RUN_TARGET_COLOR_MATCH:
    def color_match_fn(row, anchor):
        return match_color_stats(anchor, load_target(row))

    run_inference(color_match_fn, f'{BASE}/outputs/target_color_match', '_target_color_match.jpg')

## G — SDXL img2img

In [ ]:
if RUN_SDXL_IMG2IMG:
    from diffusers import AutoPipelineForImage2Image, DPMSolverMultistepScheduler

    pipe_sdxl = AutoPipelineForImage2Image.from_pretrained(
        SDXL_IMG2IMG_MODEL,
        torch_dtype=torch.float16,
        variant='fp16',
        use_safetensors=True,
    )
    pipe_sdxl.scheduler = DPMSolverMultistepScheduler.from_config(pipe_sdxl.scheduler.config)
    pipe_sdxl.enable_model_cpu_offload()

    def sdxl_img2img_fn(row, anchor):
        return pipe_sdxl(
            prompt=prompt_for(row),
            negative_prompt=NEGATIVE,
            image=anchor,
            strength=SDXL_STRENGTH,
            guidance_scale=SDXL_GUIDANCE,
            num_inference_steps=SDXL_STEPS,
        ).images[0]

    print('Running SDXL img2img...')
    run_inference(sdxl_img2img_fn, f'{BASE}/outputs/sdxl_img2img', '_sdxl_img2img.jpg')
    del pipe_sdxl
    free_gpu()
    print('GPU cleared.')

## H — SDXL ControlNet + Canny

In [ ]:
if RUN_SDXL_CONTROLNET:
    from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, UniPCMultistepScheduler

    controlnet = ControlNetModel.from_pretrained(
        SDXL_CONTROLNET_MODEL,
        torch_dtype=torch.float16,
        variant='fp16',
        use_safetensors=True,
    )
    pipe_sdxl_cn = StableDiffusionXLControlNetPipeline.from_pretrained(
        SDXL_IMG2IMG_MODEL,
        controlnet=controlnet,
        torch_dtype=torch.float16,
        variant='fp16',
        use_safetensors=True,
    )
    pipe_sdxl_cn.scheduler = UniPCMultistepScheduler.from_config(pipe_sdxl_cn.scheduler.config)
    pipe_sdxl_cn.enable_model_cpu_offload()

    def sdxl_controlnet_fn(row, anchor):
        return pipe_sdxl_cn(
            prompt=prompt_for(row),
            negative_prompt=NEGATIVE,
            image=get_canny(anchor),
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            controlnet_conditioning_scale=SDXL_COND_SCALE,
        ).images[0]

    print('Running SDXL ControlNet + Canny...')
    run_inference(sdxl_controlnet_fn, f'{BASE}/outputs/sdxl_controlnet_canny', '_sdxl_controlnet.jpg')
    del pipe_sdxl_cn, controlnet
    free_gpu()
    print('GPU cleared.')

## Add these models to `03_evaluate.ipynb`

In [ ]:
EXPLORATION_MODELS = {
    'Oracle Target Copy': (f'{BASE}/outputs/oracle_target_copy', '_oracle.jpg'),
    'Target Color Match': (f'{BASE}/outputs/target_color_match', '_target_color_match.jpg'),
    'SDXL img2img': (f'{BASE}/outputs/sdxl_img2img', '_sdxl_img2img.jpg'),
    'SDXL ControlNet': (f'{BASE}/outputs/sdxl_controlnet_canny', '_sdxl_controlnet.jpg'),
}

for model_name, (out_dir, suffix) in EXPLORATION_MODELS.items():
    print(f"'{model_name}': (f'{{BASE}}/outputs/{os.path.basename(out_dir)}', '{suffix}'),")

## Side-by-side preview

In [ ]:
import matplotlib.pyplot as plt

row = eval_df.iloc[0]
anchor = Image.open(image_path(row['src_anchor'])).convert('RGB')
gt = Image.open(image_path(row['tgt_name'])).convert('RGB')
panels = [('Anchor (source)', anchor), ('Ground Truth', gt)]

for name, (out_dir, suffix) in EXPLORATION_MODELS.items():
    p = os.path.join(out_dir, generated_name(row, suffix))
    if os.path.exists(p):
        panels.append((name, Image.open(p).convert('RGB')))

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
if len(panels) == 1:
    axes = [axes]
for ax, (title, img) in zip(axes, panels):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle(f"{row['location']} -> {row['tgt_tod']} / {row['tgt_weather']}", y=1.02)
plt.tight_layout()
plt.show()